# Training Mask R-CNN (Detectron2) on SUN RGB-D

The same architecture (Mask R-CNN, ResNet-50 + FPN), but implemented in **Detectron2** —
the reference library by the authors of Mask R-CNN (FAIR). Comparing the two
implementations on identical subsets provides extra material for chapter 2: how
much the result depends on implementation details and the training recipe rather
than the architecture alone.

**Dependencies (pinned):** Detectron2 built from source at commit
`a2f4a87` (`main` of 2026-08-19, the code the std10 run used) against the
torch / torchvision preinstalled in Colab; the build takes ~5 min once per session.

**Run switches** (next to `CONFIG`): `BACKBONE` (Model Zoo config: `R_50_FPN_3x`
| `R_101_FPN_3x` | `X_101_32x8d_FPN_3x`), `EPOCHS` (the LR decay epochs scale
with it), `MIN_SIZE_TRAIN` (multi-scale training sizes) and `REPEAT_THRESH`
(`RepeatFactorTrainingSampler` for rare classes). `AMP` (mixed precision, on by default). Each run writes to
`v2_maskrcnn_detectron2_{CONFIG}_{BACKBONE}[_suffixes]` — the dataset version is
part of the name, so runs on dataset v1 are never resumed by mistake.

In [ ]:
# install dependencies (pinned). Detectron2 has no wheels for current torch, so it is
# built from source at a fixed commit; --no-build-isolation builds it against the
# torch preinstalled in Colab instead of a fresh one
DETECTRON2_COMMIT = "a2f4a8771ab77e8411c26b27f24f9489a28a2453"
try:
    import detectron2
except ImportError:
    !pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"

In [ ]:
from pathlib import Path
import json
import os

import random, cv2
import matplotlib.pyplot as plt
import pandas as pd

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog, build_detection_test_loader
from detectron2.engine import DefaultTrainer, DefaultPredictor, hooks
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.checkpoint import DetectionCheckpointer

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
CONFIG = "std10"          # "std10" (10 classes) | "min400" (20) | "min100" (41)
SPLIT_SUBDIR = {"std10": "splits", "min400": "splits_min400",
                "min100": "splits_min100"}[CONFIG]

# run variant switches; the defaults give the baseline run
BACKBONE = "R_50_FPN_3x"  # Model Zoo: "R_50_FPN_3x" | "R_101_FPN_3x" | "X_101_32x8d_FPN_3x"
EPOCHS   = 12             # LR decay epochs scale with it (at 2/3 and 11/12 of the run)
MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)   # shorter side, random choice per image
                          # (Model Zoo multi-scale default); (800,) = single scale
REPEAT_THRESH = None      # None = uniform sampling; e.g. 0.001 -> RepeatFactorTrainingSampler
                          # (images with rare classes are repeated within an epoch)

assert BACKBONE in ("R_50_FPN_3x", "R_101_FPN_3x", "X_101_32x8d_FPN_3x"), BACKBONE
MIN_SIZE_TRAIN = tuple(MIN_SIZE_TRAIN)
AMP = True                # mixed precision (~1.5x faster on a T4)

_DEFAULT_SIZES = (640, 672, 704, 736, 768, 800)
DATA_VERSION = "v2"       # dataset version; part of the run name, so v1 runs on Drive
                          # are never resumed by mistake
RUN_NAME = f"{DATA_VERSION}_maskrcnn_detectron2_{CONFIG}_{BACKBONE}"
if EPOCHS != 12:
    RUN_NAME += f"_e{EPOCHS}"
if MIN_SIZE_TRAIN != _DEFAULT_SIZES:
    lo, hi = min(MIN_SIZE_TRAIN), max(MIN_SIZE_TRAIN)
    RUN_NAME += f"_s{lo}" if lo == hi else f"_s{lo}-{hi}"
if REPEAT_THRESH is not None:
    RUN_NAME += f"_rfs{REPEAT_THRESH:g}"
if not AMP:
    RUN_NAME += "_fp32"

DRIVE_DIR = Path("/content/drive/MyDrive/Магістратура/Be be be")   # project folder on Drive
DATA_ZIP  = DRIVE_DIR / "sunrgbd_data.zip"
DATA_ROOT = Path("/content/sunrgbd")
OUT_DIR   = DRIVE_DIR / "vkr_runs" / RUN_NAME
SPLIT_DIR = DATA_ROOT / SPLIT_SUBDIR

# unpack once per Colab session (the zip is stored uncompressed, ~5 min from Drive).
# A marker file is written only after unzip succeeds, so an interrupted or failed
# extraction (session drop, half-synced zip) is redone instead of silently reused.
import subprocess
UNPACK_OK = DATA_ROOT / ".unpacked_ok"
if not (UNPACK_OK.is_file() and UNPACK_OK.read_text() == str(DATA_ZIP.stat().st_size)):
    assert DATA_ZIP.is_file(), f"archive not found on Drive: {DATA_ZIP}"
    print("Unpacking archive...")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", str(DATA_ZIP), "-d", str(DATA_ROOT)], check=True)
    UNPACK_OK.write_text(str(DATA_ZIP.stat().st_size))

OUT_DIR.mkdir(parents=True, exist_ok=True)
for s in ("train", "val", "test"):
    assert (SPLIT_DIR / f"instances_{s}.json").is_file(), f"missing instances_{s}.json"

# the archive must hold dataset v2 (proper polygon clipping, recovered names,
# leak-free stratified split); v1 runs live in other Drive folders
_ver = json.loads((SPLIT_DIR / "dataset_version.json").read_text())["version"]
assert _ver == DATA_VERSION, f"archive holds dataset {_ver}, expected {DATA_VERSION}: re-upload sunrgbd_data.zip"

# every image of the selected config must be on the local disk
_missing = [im["file_name"] for s in ("train", "val", "test")
            for im in json.loads((SPLIT_DIR / f"instances_{s}.json").read_text())["images"]
            if not (DATA_ROOT / im["file_name"]).is_file()]
assert not _missing, (f"{len(_missing)} images missing under {DATA_ROOT} (e.g. {_missing[0]}): "
                      f"delete {UNPACK_OK} (or restart the runtime) and rerun this cell")
print("Config:", CONFIG, "| run:", RUN_NAME)
print("Data:", DATA_ROOT)
print("Outputs:", OUT_DIR)

In [ ]:
SEED         = 42
BATCH_SIZE   = 4
NUM_WORKERS  = 2
BASE_LR      = 0.005
LR_DECAY_EPOCHS = [round(EPOCHS * 2 / 3), round(EPOCHS * 11 / 12)]   # LR ÷ 10 at these
                              # epochs: [8, 11] for 12 epochs
WARMUP_ITERS = 500
CKPT_EVERY_ITERS = 500        # mid-epoch checkpoints on Drive (~10 min on a T4);
                              # a session drop costs at most this many iterations
SCORE_THRESH_TEST = 0.05      # unified prediction protocol (02/03/04): score >= 0.05,
MAX_DETS          = 100       # at most 100 detections per image
SCORE_THRESH_VIS  = 0.5       # confidence threshold for visualizations only

_train = json.loads((SPLIT_DIR / "instances_train.json").read_text())
NUM_CLASSES = len(_train["categories"])   # no background class in Detectron2
n_train = len(_train["images"])
ITERS_PER_EPOCH = (n_train + BATCH_SIZE - 1) // BATCH_SIZE   # nominal epoch (with RFS the
                                                             # schedule keeps these units)
MAX_ITER = EPOCHS * ITERS_PER_EPOCH
STEPS = tuple(e * ITERS_PER_EPOCH for e in LR_DECAY_EPOCHS)
print(f"{NUM_CLASSES} classes; train: {n_train} images, "
      f"{ITERS_PER_EPOCH} iterations/epoch, {MAX_ITER} iterations total, "
      f"LR decay at {STEPS}")

---
## Dataset

Detectron2 works with a dataset registry. `register_coco_instances` reads our
COCO files; the image paths inside are relative, so the `DATA_ROOT` root is
passed in. The COCO category ids (1…N, N = number of classes of the selected
`CONFIG`) are mapped by the library itself to contiguous indices 0…N−1; the
saved predictions are mapped back to the COCO ids.

In [ ]:
for s in ("train", "val", "test"):
    name = f"sunrgbd_{s}"
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name); MetadataCatalog.remove(name)
    register_coco_instances(name, {}, str(SPLIT_DIR / f"instances_{s}.json"), str(DATA_ROOT))

# loading the dicts also fills in metadata (thing_classes) lazily
for s in ("train", "val", "test"):
    print(f"{s:5s}: {len(DatasetCatalog.get(f'sunrgbd_{s}'))} images")

meta = MetadataCatalog.get("sunrgbd_train")
print("Classes:", meta.thing_classes)

In [ ]:
# visual check of the annotations using Detectron2 tooling

rng = random.Random(SEED)
samples = rng.sample(DatasetCatalog.get("sunrgbd_train"), 3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, d in zip(axes, samples):
    img = cv2.imread(d["file_name"])[:, :, ::-1]
    vis = Visualizer(img, metadata=meta, scale=0.8)
    out = vis.draw_dataset_dict(d)
    ax.imshow(out.get_image()); ax.axis("off")
plt.tight_layout(); plt.show()

---
## Configuration

The base is the Model Zoo config selected by `BACKBONE` (default
`mask_rcnn_R_50_FPN_3x.yaml`: Mask R-CNN, ResNet-50 + FPN, weights pre-trained
on COCO, the "3x" schedule; also `R_101_FPN_3x` and `X_101_32x8d_FPN_3x`). The
number of classes, the batch size and the training schedule are changed; the
Model Zoo recipe already trains multi-scale (shorter side 640–800,
`MIN_SIZE_TRAIN`) and tests at 800. Optionally (`REPEAT_THRESH`) the
`RepeatFactorTrainingSampler` oversamples images that contain rare classes
(LVIS-style repeat factor `max(1, sqrt(t / f_c))`). The test-time score
threshold (0.05) and detection limit (100) are set explicitly.

In [ ]:
ZOO_CFG = f"COCO-InstanceSegmentation/mask_rcnn_{BACKBONE}.yaml"
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(ZOO_CFG))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(ZOO_CFG)

cfg.DATASETS.TRAIN = ("sunrgbd_train",)
cfg.DATASETS.TEST  = ("sunrgbd_val",)          # in-training evaluation — on the validation set
cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
cfg.INPUT.MIN_SIZE_TRAIN = MIN_SIZE_TRAIN       # random choice per image ("choice" sampling)
if REPEAT_THRESH is not None:
    cfg.DATALOADER.SAMPLER_TRAIN = "RepeatFactorTrainingSampler"
    cfg.DATALOADER.REPEAT_THRESHOLD = REPEAT_THRESH

cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST   # unified protocol; never
cfg.TEST.DETECTIONS_PER_IMAGE = MAX_DETS                    # changed on this global cfg
cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
cfg.SOLVER.BASE_LR = BASE_LR
cfg.SOLVER.MAX_ITER = MAX_ITER
cfg.SOLVER.STEPS = STEPS
cfg.SOLVER.WARMUP_ITERS = WARMUP_ITERS
cfg.SOLVER.AMP.ENABLED = AMP                    # mixed-precision training
cfg.SOLVER.CHECKPOINT_PERIOD = CKPT_EVERY_ITERS     # checkpoint inside the epoch (resume point)
cfg.TEST.EVAL_PERIOD = ITERS_PER_EPOCH              # evaluation every epoch
cfg.SEED = SEED
cfg.OUTPUT_DIR = str(OUT_DIR)

print(f"{ZOO_CFG}: MAX_ITER={cfg.SOLVER.MAX_ITER}, STEPS={cfg.SOLVER.STEPS}, "
      f"MIN_SIZE_TRAIN={cfg.INPUT.MIN_SIZE_TRAIN}, sampler={cfg.DATALOADER.SAMPLER_TRAIN}, "
      f"eval every {cfg.TEST.EVAL_PERIOD} iterations")

---
## Training

`DefaultTrainer` + two additions:

* **COCOEvaluator on the validation set** every epoch — the same metrics
  (mask AP) as in notebook 02;
* **BestCheckpointer** — saves `model_best.pth` by the highest `segm/AP`. The
  stock hook forgets the best score on every resume, so the first evaluation
  after a resume overwrote `model_best.pth` even with a worse model (in the std10
  run the true best, 24.78 @ iteration 15960, was replaced by the final model,
  24.54). `ResumableBestCheckpointer` restores the best score from
  `metrics.json` before training continues.

`resume_or_load(resume=True)` allows training to continue from the last
checkpoint after a Colab session drop — just rerun the cells from the top.
Checkpoints (`model_*.pth`, full state: weights + optimizer + LR scheduler +
iteration) are written every `CKPT_EVERY_ITERS` iterations, not once per
epoch, so at most ~10 minutes of compute are lost; only the 2 most recent
ones are kept on Drive (each is ~330 MB). `model_best.pth` and
`model_final.pth` are never deleted. If the resumed iteration is already at
`MAX_ITER` (the notebook is rerun after training finished), `trainer.train()`
is skipped entirely, so nothing is re-evaluated or overwritten.

In [ ]:
import math


def best_from_metrics(metrics_file, key, before_iter=None):
    """(max value of `key`, its iteration) over Detectron2's metrics.json, or None.

    Only evaluations at iterations < before_iter count: later records belong to a
    session that dropped before its next checkpoint and will be recomputed.
    """
    best = None
    if not Path(metrics_file).is_file():
        return best
    with open(metrics_file) as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:      # a line cut off by a session drop
                continue
            val, it = rec.get(key), rec.get("iteration")
            if (val is None or it is None or not math.isfinite(val)
                    or (before_iter is not None and it >= before_iter)):
                continue
            if best is None or val > best[0]:
                best = (float(val), int(it))
    return best


class ResumableBestCheckpointer(hooks.BestCheckpointer):
    """BestCheckpointer (mode="max") that keeps the best score across resumes.

    The stock hook starts every session with best_metric=None, so the first evaluation
    after a resume always overwrote model_best.pth, even with a worse model.
    """

    def before_train(self):
        save_dir = Path(self._checkpointer.save_dir)
        best = best_from_metrics(save_dir / "metrics.json", self._val_metric,
                                 before_iter=self.trainer.start_iter)
        if best is not None and (save_dir / f"{self._file_prefix}.pth").is_file():
            self.best_metric, self.best_iter = best
            print(f"Restored best {self._val_metric} = {self.best_metric:.4f} "
                  f"@ iteration {self.best_iter}")


class Trainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "eval_val")
        return COCOEvaluator(dataset_name, output_dir=output_folder)

    def build_hooks(self):
        # same hooks as DefaultTrainer, but keep only the 2 most recent periodic
        # checkpoints on Drive (model_best.pth / model_final.pth are not affected)
        ret = super().build_hooks()
        for i, h in enumerate(ret):
            if isinstance(h, hooks.PeriodicCheckpointer):
                ret[i] = hooks.PeriodicCheckpointer(
                    self.checkpointer, self.cfg.SOLVER.CHECKPOINT_PERIOD, max_to_keep=2)
        return ret


trainer = Trainer(cfg)
trainer.register_hooks([ResumableBestCheckpointer(
    cfg.TEST.EVAL_PERIOD, trainer.checkpointer, "segm/AP", mode="max")])
trainer.resume_or_load(resume=True)
if trainer.start_iter >= trainer.max_iter:
    # rerun after training finished: trainer.train() would still run the final
    # evaluation in after_train and could overwrite model_best.pth — skip it entirely
    print(f"Training already finished (resume iteration {trainer.start_iter} >= "
          f"MAX_ITER {trainer.max_iter}): trainer.train() skipped, nothing overwritten.")
else:
    trainer.train()

In [ ]:
# training curves from metrics.json
lines = []
with open(OUT_DIR / "metrics.json") as f:
    for line in f:
        try:
            lines.append(json.loads(line))
        except json.JSONDecodeError:
            continue
m = pd.DataFrame(lines)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

loss = m.dropna(subset=["total_loss"])
axes[0].plot(loss.iteration, loss.total_loss, lw=0.7, color="#2f6f9f")
axes[0].set_xlabel("Iteration"); axes[0].set_title("Total loss"); axes[0].grid(alpha=0.3)

for col, lbl in [("loss_cls", "cls"), ("loss_box_reg", "box"),
                 ("loss_mask", "mask"), ("loss_rpn_cls", "rpn_cls"),
                 ("loss_rpn_loc", "rpn_loc")]:
    if col in m:
        sub = m.dropna(subset=[col])
        axes[1].plot(sub.iteration, sub[col], lw=0.7, label=lbl)
axes[1].set_xlabel("Iteration"); axes[1].set_title("Loss components")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

if "segm/AP" in m:
    ap = m.dropna(subset=["segm/AP"])
    axes[2].plot(ap.iteration, ap["segm/AP"], "-o", ms=3, color="#c0392b", label="mask AP")
    if "bbox/AP" in m:
        ap2 = m.dropna(subset=["bbox/AP"])
        axes[2].plot(ap2.iteration, ap2["bbox/AP"], "-o", ms=3, color="#2f6f9f", label="box AP")
    axes[2].set_xlabel("Iteration"); axes[2].set_title("Validation quality (val)")
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Final evaluation on the test set

Evaluates the model that is **best by validation**, loaded explicitly from
`model_best.pth` (never the weights left in memory after training). Predictions
are collected with a plain inference loop and scored with pycocotools COCOeval —
the same code path as notebooks 02/04 (Detectron2's `COCOEvaluator` is used only
for the in-training validation).

### Unified prediction and timing protocol (identical in notebooks 02/03/04)

* **Predictions for AP.** The model selected on validation predicts the val and
  test sets with score threshold **0.05** (torchvision `box_score_thresh`,
  Detectron2 `SCORE_THRESH_TEST`, YOLO `conf`) and at most **100** detections
  per image, in FP32; masks are binarized at 0.5 at the original image
  resolution. They are saved to `OUT_DIR/val_predictions.json` and
  `OUT_DIR/test_predictions.json` in the common COCO results format: int
  `image_id`, `category_id` = COCO category id, `bbox` = `[x, y, w, h]`,
  `segmentation` = RLE with `counts` as str, `score`. These files are the input
  of `utils/evaluate_predictions.py`, the single evaluator used to compare
  models; `test_metrics.csv` / `test_per_class_ap.csv` hold pycocotools
  COCOeval values as fractions (0–1).
* **Inference timing.** The same 100 test images for every model (the first 100
  test image ids in ascending order), decoded beforehand into RGB uint8
  H×W×3 numpy arrays in memory. The timed region is the model's own
  preprocessing + forward pass + postprocessing up to the final binary masks at
  the original resolution (score ≥ 0.05, ≤ 100 detections), FP32, batch 1;
  10 warmup runs, then 100 timed runs, with `torch.cuda.synchronize()` before
  the clock is stopped. `timing.json` stores the mean / std / median latency
  (ms) and FPS; only numbers measured on the same GPU (T4) are compared.

In [ ]:
import numpy as np
import torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from detectron2.evaluation.coco_evaluation import instances_to_coco_json

best = OUT_DIR / "model_best.pth"
assert best.is_file(), "model_best.pth not found - has training finished?"
model = Trainer.build_model(cfg)                      # cfg: score >= 0.05, <= 100 dets
extra = DetectionCheckpointer(model).load(str(best))
model.eval()
print(f"Weights: {best.name} (iteration {extra.get('iteration')})")


@torch.no_grad()
def predict_coco(model, dataset_name):
    """Predictions for a registered split in the common COCO results format."""
    to_coco_id = {v: k for k, v in
                  MetadataCatalog.get(dataset_name).thing_dataset_id_to_contiguous_id.items()}
    results = []
    for inputs in build_detection_test_loader(cfg, dataset_name):
        for inp, out in zip(inputs, model(inputs)):
            # masks are pasted at the original resolution, RLE counts as str, bbox xywh
            for r in instances_to_coco_json(out["instances"].to("cpu"), inp["image_id"]):
                r["image_id"] = int(r["image_id"])
                r["category_id"] = to_coco_id[r["category_id"]]
                results.append(r)
    return results


def coco_metrics(coco_gt, results):
    """pycocotools COCOeval -> (metrics as fractions, per-class mask AP), as in 02/04."""
    names = {c["id"]: c["name"] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    coco_dt = coco_gt.loadRes([dict(r) for r in results])   # loadRes adds keys in place
    metrics, per_class = {}, {}
    for iou_type in ("segm", "bbox"):
        ev = COCOeval(coco_gt, coco_dt, iou_type)
        ev.evaluate(); ev.accumulate(); ev.summarize()
        for k, stat in enumerate(("AP", "AP50", "AP75", "APs", "APm", "APl")):
            metrics[f"{iou_type}_{stat}"] = ev.stats[k]
        if iou_type == "segm":
            # per-class AP: precision has shape [IoU, recall, class, area, maxDet]
            prec = ev.eval["precision"]
            for k, cat_id in enumerate(ev.params.catIds):
                p = prec[:, :, k, 0, -1]
                per_class[names[cat_id]] = float(p[p > -1].mean()) if (p > -1).any() else float("nan")
    return metrics, per_class


# Predictions already saved for this model_best.pth are reused, so a re-run of a finished
# run (e.g. only for the timing cell) does not change the reported numbers; REPREDICT = True
# forces new predictions.
REPREDICT = False


def predictions(split):
    path = OUT_DIR / f"{split}_predictions.json"
    if not REPREDICT and path.is_file() and path.stat().st_mtime > best.stat().st_mtime:
        print(f"{split}: reusing saved predictions {path.name}")
        with open(path) as f:
            return json.load(f)
    results = predict_coco(model, f"sunrgbd_{split}")
    with open(path, "w") as f:
        json.dump(results, f)
    print(f"{split}: predictions saved: {path}")
    return results


# validation predictions of the selected model (the same protocol as test)
val_results = predictions("val")
val_metrics, _ = coco_metrics(COCO(str(SPLIT_DIR / "instances_val.json")), val_results)
print(f"\nval mask AP = {val_metrics['segm_AP']:.4f}")

test_results = predictions("test")
test_metrics, test_per_class = coco_metrics(COCO(str(SPLIT_DIR / "instances_test.json")),
                                            test_results)

summary = pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}).round(4)
summary.to_csv(OUT_DIR / "test_metrics.csv")
print("\nFinal metrics (test):"); display(summary)

per_class_df = (pd.Series(test_per_class, name="mask AP").sort_values(ascending=False)
                .to_frame().round(4))
per_class_df.to_csv(OUT_DIR / "test_per_class_ap.csv")
print("Per-class mask AP:"); display(per_class_df)

In [ ]:
# extra test metrics from the saved predictions (no GPU needed):
# per-class precision / recall / F1, confusion matrix, mean IoU of matched masks
import numpy as np, json
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

CONF_THRESH = 0.5   # confidence threshold (P/R at fixed conf, like YOLO reports)
IOU_THRESH  = 0.5   # mask IoU threshold for matching predictions to GT

coco_gt = COCO(str(SPLIT_DIR / "instances_test.json"))
PRED_FILE = OUT_DIR / "test_predictions.json"
preds = [p for p in json.loads(PRED_FILE.read_text()) if p["score"] >= CONF_THRESH]

cat_ids   = sorted(coco_gt.getCatIds())
cat_names = [coco_gt.loadCats(c)[0]["name"] for c in cat_ids]
cat_idx   = {c: i for i, c in enumerate(cat_ids)}
K = len(cat_ids)

# confusion matrix: rows = GT classes (+ background), cols = predicted (+ missed)
conf = np.zeros((K + 1, K + 1), dtype=int)
tp, fp, fn = np.zeros(K, int), np.zeros(K, int), np.zeros(K, int)
matched_ious = []

preds_by_img = {}
for p in preds:
    preds_by_img.setdefault(p["image_id"], []).append(p)

for img_id in coco_gt.getImgIds():
    gts = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_id, iscrowd=False))
    dts = sorted(preds_by_img.get(img_id, []), key=lambda p: -p["score"])
    gt_rles = [coco_gt.annToRLE(a) for a in gts]
    dt_rles = [d["segmentation"] for d in dts]
    ious = (mask_utils.iou(dt_rles, gt_rles, [0] * len(gt_rles))
            if gt_rles and dt_rles else np.zeros((len(dt_rles), len(gt_rles))))

    # greedy class-agnostic matching by descending score
    gt_used = set()
    for di, d in enumerate(dts):
        best_gi, best_iou = -1, IOU_THRESH
        for gi in range(len(gts)):
            if gi not in gt_used and ious[di, gi] >= best_iou:
                best_gi, best_iou = gi, ious[di, gi]
        dk = cat_idx[d["category_id"]]
        if best_gi >= 0:
            gt_used.add(best_gi)
            gk = cat_idx[gts[best_gi]["category_id"]]
            conf[gk, dk] += 1
            matched_ious.append(best_iou)
            if gk == dk:
                tp[dk] += 1
            else:                     # localized but misclassified
                fp[dk] += 1; fn[gk] += 1
        else:                         # no GT match -> false positive on background
            conf[K, dk] += 1
            fp[dk] += 1
    for gi, g in enumerate(gts):      # unmatched GT -> missed object
        if gi not in gt_used:
            gk = cat_idx[g["category_id"]]
            conf[gk, K] += 1
            fn[gk] += 1

prec = tp / np.maximum(tp + fp, 1)
rec  = tp / np.maximum(tp + fn, 1)
f1   = 2 * prec * rec / np.maximum(prec + rec, 1e-9)

table = pd.DataFrame({"precision": prec, "recall": rec, "F1": f1,
                      "TP": tp, "FP": fp, "FN": fn}, index=cat_names)
mp = tp.sum() / max(tp.sum() + fp.sum(), 1)
mr = tp.sum() / max(tp.sum() + fn.sum(), 1)
table.loc["micro avg"] = [mp, mr, 2 * mp * mr / max(mp + mr, 1e-9),
                          tp.sum(), fp.sum(), fn.sum()]
table.loc["macro avg"] = [prec.mean(), rec.mean(), f1.mean(),
                          tp.sum(), fp.sum(), fn.sum()]
table = table.round(4)
table.to_csv(OUT_DIR / "test_pr_f1.csv")
print(f"conf >= {CONF_THRESH}, mask IoU >= {IOU_THRESH}")
print(f"Mean IoU of matched masks: {np.mean(matched_ious):.4f}")
display(table)

# confusion matrix normalized by GT row
norm = conf / np.maximum(conf.sum(axis=1, keepdims=True), 1)
row_lbl = cat_names + ["background"]
col_lbl = cat_names + ["missed"]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(K + 1)); ax.set_xticklabels(col_lbl, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(K + 1)); ax.set_yticklabels(row_lbl, fontsize=8)
for i in range(K + 1):
    for j in range(K + 1):
        if conf[i, j]:
            ax.text(j, i, conf[i, j], ha="center", va="center", fontsize=7,
                    color="white" if norm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Confusion matrix (mask IoU >= {IOU_THRESH}, conf >= {CONF_THRESH})")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# qualitative examples on test images (a cfg copy with the visualization threshold:
# the global cfg keeps the 0.05 threshold of the unified protocol)
cfg_vis = cfg.clone()
cfg_vis.MODEL.WEIGHTS = str(best)
cfg_vis.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_VIS
predictor = DefaultPredictor(cfg_vis)

test_dicts = rng.sample(DatasetCatalog.get("sunrgbd_test"), 4)
fig, axes = plt.subplots(len(test_dicts), 2, figsize=(12, 5 * len(test_dicts)))
for row, d in enumerate(test_dicts):
    img = cv2.imread(d["file_name"])[:, :, ::-1]

    vis_gt = Visualizer(img, metadata=meta, scale=0.8).draw_dataset_dict(d)
    axes[row, 0].imshow(vis_gt.get_image()); axes[row, 0].axis("off")
    axes[row, 0].set_title("Ground truth", fontsize=9)

    out = predictor(img[:, :, ::-1])
    vis_pr = Visualizer(img, metadata=meta, scale=0.8).draw_instance_predictions(
        out["instances"].to("cpu"))
    axes[row, 1].imshow(vis_pr.get_image()); axes[row, 1].axis("off")
    axes[row, 1].set_title(f"Mask R-CNN / Detectron2 (score>{SCORE_THRESH_VIS})", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "qualitative_test.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Inference timing

End-to-end single-image latency under the unified protocol described above the
final evaluation: the same 100 test images as RGB uint8 arrays in memory,
preprocessing + forward pass + postprocessing up to the final masks at the
original resolution, score threshold 0.05, FP32, batch 1, 10 warmup + 100 timed
runs, GPU-synchronized.

In [ ]:
# inference timing — the unified protocol (same cell logic in notebooks 02/03/04)
import time
import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO

N_WARMUP, N_RUNS = 10, 100
coco_t = COCO(str(SPLIT_DIR / "instances_test.json"))
timing_ids = sorted(coco_t.getImgIds())[:N_RUNS]       # the same 100 images in 02/03/04
# decoded RGB uint8 HxWx3 arrays in memory: disk I/O and JPEG decoding are not timed
arrs = [np.array(Image.open(DATA_ROOT / coco_t.loadImgs(i)[0]["file_name"]).convert("RGB"))
        for i in timing_ids]
MODEL_TAG = RUN_NAME
cfg_t = cfg.clone()                      # a copy: the global cfg is never mutated
cfg_t.MODEL.WEIGHTS = str(best)
cfg_t.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
cfg_t.TEST.DETECTIONS_PER_IMAGE = MAX_DETS
predictor = DefaultPredictor(cfg_t)


def predict(rgb):
    """RGB uint8 HxWx3 -> final binary masks at the original resolution."""
    out = predictor(np.ascontiguousarray(rgb[:, :, ::-1]))   # DefaultPredictor takes BGR;
    return out["instances"].pred_masks                         # resize, forward, paste masks


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

for a in arrs[:N_WARMUP]:
    predict(a)
_sync()
times = []
for a in arrs:
    _sync(); t0 = time.perf_counter()
    predict(a)
    _sync(); times.append(time.perf_counter() - t0)

times = np.array(times) * 1000
timing = {"model": MODEL_TAG,
          "device": (torch.cuda.get_device_name(0)
                     if torch.cuda.is_available() else "cpu"),
          "n_runs": len(times), "n_images": len(arrs),
          "score_thresh": SCORE_THRESH_TEST,
          "mean_ms": float(times.mean()), "std_ms": float(times.std()),
          "median_ms": float(np.median(times)),
          "fps": float(1000.0 / times.mean())}
(OUT_DIR / "timing.json").write_text(json.dumps(timing, indent=2))
print(json.dumps(timing, indent=2))